In [ ]:

#@title 1. Configuración del Entorno
import os

# Clonar el repositorio si no existe
if not os.path.exists("Signal-Analysis-Vol-I"):
    !git clone https://github.com/ecundir/Signal-Analysis-Vol-I.git

# Cambiar a la carpeta del capítulo
%cd Signal-Analysis-Vol-I/capitulo_6
print("✅ Entorno configurado correctamente.")

Para aprovechar al máximo el Laboratorio Vivo en Google Colab, comienza ejecutando la celda de código para desplegar la interfaz interactiva. Utiliza el deslizador Orden (n) para observar cómo un valor más alto incrementa la pendiente de caída (selectividad) del filtro, y ajusta la Frecuencia de Corte para desplazar la ventana de operación sobre el espectro. Es fundamental que observes simultáneamente ambas gráficas: mientras que en la superior verás cómo el filtro rechaza las frecuencias del "canal vecino", en la inferior notarás cómo estos cambios afectan la estabilidad del tiempo de tránsito de la señal.

Como ejercicio práctico, selecciona el filtro Chebyshev y aumenta gradualmente el Ripple (dB); notarás que, aunque el filtro se vuelve "más agresivo" para cortar la interferencia, la banda de paso pierde uniformidad y el retardo de grupo muestra picos pronunciados. Compara esto con el filtro de Bessel, el cual, sin importar el orden que elijas, mantendrá siempre una respuesta de retardo plana en la zona de paso. Este contraste visual te permitirá entender por qué en ingeniería de radio la elección de un filtro es siempre un compromiso entre proteger el espectro y mantener la integridad de los bits.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import interact, FloatSlider, IntSlider

# Configuración inicial de estilo
plt.rcParams.update({"font.family": "serif", "mathtext.fontset": "cm"})

def plot_interactive_filters(n=5, f_corte=15000, rp=1):
    fs = 100e3  # Frecuencia de muestreo fija
    w_limit = 40000 # Límite de visualización en Hz

    # 1. Diseño de Filtros (Forma SOS para estabilidad)
    sos_but = signal.butter(n, f_corte, btype='low', fs=fs, output='sos')
    sos_cheb = signal.cheby1(n, rp, f_corte, btype='low', fs=fs, output='sos')
    sos_bess = signal.bessel(n, f_corte, btype='low', norm='phase', fs=fs, output='sos')

    # 2. Cálculo de Respuesta en Frecuencia
    w, h_but = signal.sosfreqz(sos_but, worN=8000, fs=fs)
    _, h_cheb = signal.sosfreqz(sos_cheb, worN=8000, fs=fs)
    _, h_bess = signal.sosfreqz(sos_bess, worN=8000, fs=fs)

    # 3. Creación de la Gráfica
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

    # --- Subtrama 1: Magnitud ---
    ax1.plot(w, 20 * np.log10(np.maximum(abs(h_but), 1e-10)), label='Butterworth (Plano)', lw=2)
    ax1.plot(w, 20 * np.log10(np.maximum(abs(h_cheb), 1e-10)), label=f'Chebyshev ({rp}dB ripple)', ls='--')
    ax1.plot(w, 20 * np.log10(np.maximum(abs(h_bess), 1e-10)), label='Bessel (Fase)', ls=':')

    ax1.axvline(f_corte, color='black', alpha=0.3, ls='--', label='f_corte')
    ax1.axvspan(20000, w_limit, color='red', alpha=0.05, label='Zona Vecino')
    ax1.set_ylabel("Magnitud (dB)", fontsize=12)
    ax1.set_ylim(-80, 5)
    ax1.set_title(r"Análisis Dinámico de la Función de Transferencia $H(j\omega)$", fontsize=15)
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper right')

    # --- Subtrama 2: Retardo de Grupo ---
    # Conversión temporal para group_delay (estabilidad local)
    for sos, label, style in [(sos_but, 'But', '-'), (sos_cheb, 'Cheb', '--'), (sos_bess, 'Bess', ':')]:
        b, a = signal.sos2tf(sos)
        w_gd, gd = signal.group_delay((b, a), w=np.linspace(0, f_corte*1.2, 500), fs=fs)
        ax2.plot(w_gd, gd * 1e6, label=label, ls=style, lw=2)

    ax2.set_ylabel(r"Retardo de Grupo ($\mu s$)", fontsize=12)
    ax2.set_xlabel("Frecuencia (Hz)", fontsize=12)
    ax2.set_xlim(0, w_limit)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Crear los controles interactivos
interact(plot_interactive_filters,
         n=IntSlider(min=1, max=15, step=1, value=5, description='Orden (n)'),
         f_corte=FloatSlider(min=5000, max=25000, step=1000, value=15000, description='F. Corte'),
         rp=FloatSlider(min=0.1, max=10, step=0.5, value=1, description='Ripple (dB)'));